# Brain tumor segmentation with nnU-Net (one clean notebook)
**Run the cells top to bottom, one at a time. Do not skip. Runtime must be GPU:** Runtime > Change runtime type > T4 GPU.

Where things live:
- **Local disk** (`/content`): data + preprocessing (fast, wiped when the session ends)
- **Google Drive**: only training checkpoints/results (survive session death)

## Step 1: Setup (≈2 min)

In [ ]:
import os, sys, subprocess
from google.colab import drive
drive.mount('/content/drive')

# ---- settings you may change -------------------------------------------------
TRAINER = 'nnUNetTrainer_100epochs'   # 100 epochs: realistic on free Colab. (250epochs is slower)
CONFIG  = '3d_fullres'
FOLD    = 0
# ------------------------------------------------------------------------------

DRIVE_DIR = '/content/drive/MyDrive/brats_nnunet_data'
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs('/content/data', exist_ok=True)
os.environ['nnUNet_raw']          = '/content/nnunet/raw'
os.environ['nnUNet_preprocessed'] = '/content/nnunet/preprocessed'
os.environ['nnUNet_results']      = DRIVE_DIR + '/nnUNet_results'
for k in ['nnUNet_raw','nnUNet_preprocessed','nnUNet_results']:
    os.makedirs(os.environ[k], exist_ok=True)

!pip install -q nnunetv2 nibabel scipy pandas matplotlib tqdm
!nvidia-smi | head -n 12
print('\nSetup done.')

## Step 2: Get the data onto local disk (≈5-10 min)
Uses your Drive copy if it exists (faster), otherwise downloads fresh.

In [ ]:
import shutil, glob
LOCAL = '/content/data/Task01_BrainTumour'
DRIVE_COPY = DRIVE_DIR + '/Task01_BrainTumour'

if os.path.exists(LOCAL + '/imagesTr'):
    print('Data already on local disk.')
elif os.path.exists(DRIVE_COPY + '/imagesTr'):
    print('Copying from Drive to local disk...')
    shutil.copytree(DRIVE_COPY, LOCAL)
else:
    print('Downloading (7 GB)...')
    !wget -q --show-progress -O /content/data/task01.tar https://msd-for-monai.s3-us-west-2.amazonaws.com/Task01_BrainTumour.tar
    !tar -xf /content/data/task01.tar -C /content/data
    !rm /content/data/task01.tar

# remove macOS junk files that break nnU-Net
!find /content/data -name '._*' -delete
n = len(glob.glob(LOCAL + '/imagesTr/*.nii.gz'))
print('Training images found:', n, '(expected 484)')
assert n == 484, 'Data not found/complete. Stop and tell Claude.'

## Step 3: Convert + preprocess for nnU-Net (≈30-60 min, just wait)

In [ ]:
!nnUNetv2_convert_MSD_dataset -i /content/data/Task01_BrainTumour
!nnUNetv2_plan_and_preprocess -d 1 --verify_dataset_integrity -c 2d 3d_fullres
print('Preprocessing done.')

## Step 4: Look at the data (≈1 min) - gives you your first README figure

In [ ]:
import numpy as np, nibabel as nib, matplotlib.pyplot as plt
raw = os.environ['nnUNet_raw'] + '/Dataset001_BrainTumour'
cases = sorted(os.path.basename(f)[:-7] for f in glob.glob(raw + '/labelsTr/*.nii.gz'))
print(len(cases), 'cases')
cid = cases[0]
imgs = [np.asarray(nib.load(f'{raw}/imagesTr/{cid}_{m:04d}.nii.gz').dataobj) for m in range(4)]
lab = np.asarray(nib.load(f'{raw}/labelsTr/{cid}.nii.gz').dataobj)
z = int(np.argmax((lab > 0).sum((0,1))))
fig, ax = plt.subplots(1, 5, figsize=(18, 3.8))
for i, n in enumerate(['FLAIR','T1w','T1gd','T2w']):
    ax[i].imshow(np.rot90(imgs[i][..., z]), cmap='gray'); ax[i].set_title(n); ax[i].axis('off')
ax[4].imshow(np.rot90(imgs[0][..., z]), cmap='gray')
ax[4].imshow(np.ma.masked_equal(np.rot90(lab[..., z]), 0), cmap='autumn', alpha=.6, vmin=1, vmax=3)
ax[4].set_title('Tumor labels'); ax[4].axis('off')
plt.tight_layout()
os.makedirs(DRIVE_DIR + '/figures', exist_ok=True)
plt.savefig(DRIVE_DIR + '/figures/eda_example.png', dpi=150)
plt.show()

## Step 5: Train (hours)
- Checkpoints save to Drive automatically. **If the session disconnects, just do Step 1 -> Step 2 -> Step 3 again, then run this cell again. It resumes where it stopped.**
- Keep the tab open. Free Colab limits GPU time per day; if it cuts you off, come back later and rerun.

In [ ]:
# Resumes automatically if a checkpoint exists, otherwise starts fresh
ck = f"{os.environ['nnUNet_results']}/Dataset001_BrainTumour/{TRAINER}__nnUNetPlans__{CONFIG}/fold_{FOLD}/checkpoint_latest.pth"
flag = '--c' if os.path.exists(ck) else ''
print('Resuming from checkpoint' if flag else 'Starting fresh training')
!nnUNetv2_train 1 $CONFIG $FOLD -tr $TRAINER $flag

## Step 6: Predict the validation cases

In [ ]:
import json, shutil
split = json.load(open(os.environ['nnUNet_preprocessed'] + '/Dataset001_BrainTumour/splits_final.json'))[FOLD]['val']
IN, OUT = '/content/val_in', f'/content/pred_{CONFIG}'
shutil.rmtree(IN, ignore_errors=True); os.makedirs(IN); os.makedirs(OUT, exist_ok=True)
for c in split:
    for m in range(4):
        shutil.copy(f'{raw}/imagesTr/{c}_{m:04d}.nii.gz', IN)
print(len(split), 'validation cases')
!nnUNetv2_predict -i $IN -o $OUT -d 1 -c $CONFIG -f $FOLD -tr $TRAINER

## Step 7: Score it (Dice + Hausdorff distance 95)

In [ ]:
import pandas as pd
from scipy import ndimage as ndi
REGIONS = {'WT': (1,2,3), 'TC': (2,3), 'ET': (3,)}   # whole tumor, tumor core, enhancing tumor

def dice(p, g):
    s = p.sum() + g.sum()
    return 1.0 if s == 0 else 2.0 * (p & g).sum() / s

def hd95(p, g, sp):
    if p.sum() == 0 and g.sum() == 0: return 0.0
    if p.sum() == 0 or g.sum() == 0: return float('nan')
    ps, gs = p & ~ndi.binary_erosion(p), g & ~ndi.binary_erosion(g)
    d = np.concatenate([ndi.distance_transform_edt(~gs, sampling=sp)[ps],
                        ndi.distance_transform_edt(~ps, sampling=sp)[gs]])
    return float(np.percentile(d, 95))

rows = []
for c in split:
    pi = nib.load(f'{OUT}/{c}.nii.gz')
    pred = np.asarray(pi.dataobj).astype(int)
    gt = np.asarray(nib.load(f'{raw}/labelsTr/{c}.nii.gz').dataobj).astype(int)
    sp = tuple(float(x) for x in pi.header.get_zooms()[:3])
    r = {'case': c}
    for k, labs in REGIONS.items():
        p, g = np.isin(pred, labs), np.isin(gt, labs)
        r[f'dice_{k}'], r[f'hd95_{k}'] = dice(p, g), hd95(p, g, sp)
    rows.append(r)
df = pd.DataFrame(rows)
df.to_csv(DRIVE_DIR + f'/nnunet_{CONFIG}_per_case.csv', index=False)
print(df.drop(columns='case').agg(['mean','std']).round(3).T)
print('Saved per-case CSV to Drive.')